In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import random

# Define the encoder network (ConvNet)
class ProtoEncoder(nn.Module):
    def __init__(self):
        super(ProtoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.encoder(x)
        return x.view(x.size(0), -1)  # Flatten output

# Few-shot dataset preparation for CIFAR-10
class FewShotDataset(Dataset):
    def __init__(self, dataset, n_way, k_shot, q_query):
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.data_by_class = self._organize_by_class()

    def _organize_by_class(self):
        data_by_class = {}
        for img, label in self.dataset:
            if label not in data_by_class:
                data_by_class[label] = []
            data_by_class[label].append(img)
        return data_by_class

    def __len__(self):
        return 1000  # Number of episodes

    def __getitem__(self, index):
        sampled_classes = random.sample(list(self.data_by_class.keys()), self.n_way)

        support_images, query_images, labels = [], [], []
        for i, class_id in enumerate(sampled_classes):
            class_images = random.sample(self.data_by_class[class_id], self.k_shot + self.q_query)
            support_images.extend(class_images[:self.k_shot])
            query_images.extend(class_images[self.k_shot:])
            labels.extend([i] * self.q_query)

        support_images = torch.stack(support_images)
        query_images = torch.stack(query_images)
        return support_images, query_images, torch.tensor(labels)

# Euclidean distance
def euclidean_distance(a, b):
    return torch.sum((a - b) ** 2, dim=-1)

# Prototypical loss function
def prototypical_loss(encoder, support_images, query_images, labels, n_way, k_shot):
    # Encode support and query images
    support_embeddings = encoder(support_images)  # (n_way * k_shot, embedding_dim)
    query_embeddings = encoder(query_images)      # (n_way * q_query, embedding_dim)

    # Reshape support embeddings to compute class prototypes
    support_embeddings = support_embeddings.view(n_way, k_shot, -1)  # (n_way, k_shot, embedding_dim)

    # Compute prototypes (mean of support embeddings for each class)
    prototypes = support_embeddings.mean(dim=1)  # (n_way, embedding_dim)

    # Calculate pairwise distances between query and prototypes
    def euclidean_distance(x, y):
        return torch.sum((x - y) ** 2, dim=-1)

    dists = torch.stack([euclidean_distance(query_embeddings, proto) for proto in prototypes], dim=1)  # (n_way * q_query, n_way)

    # Ensure labels match query size
    labels = labels.view(-1)  # Flatten labels to match query size

    # Compute cross-entropy loss
    return F.cross_entropy(-dists, labels)


# Training loop (modified to return the trained model)
def train_prototypical_network(n_way=5, k_shot=5, q_query=15, epochs=5, device='cuda'):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(FewShotDataset(cifar10_train, n_way, k_shot, q_query), batch_size=1, shuffle=True)

    model = ProtoEncoder().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        total_loss = 0
        for i, (support_images, query_images, labels) in enumerate(train_loader):
            support_images, query_images, labels = support_images.squeeze(0).to(device), query_images.squeeze(0).to(device), labels.to(device)

            optimizer.zero_grad()
            loss = prototypical_loss(model, support_images, query_images, labels, n_way, k_shot)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch + 1}/{epochs}], Step [{i + 1}], Loss: {loss.item():.4f}")

        print(f"Epoch [{epoch + 1}/{epochs}], Average Loss: {total_loss / len(train_loader):.4f}")

    print("Training completed.")
    return model  # Return the trained model


if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Train the model and store it in a variable
    trained_model = train_prototypical_network(device=device)
    
    # Save the trained model to disk if needed
    torch.save(trained_model.state_dict(), "proto_network.pth")
    print("Model saved successfully.")


/opt/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Epoch [1/5], Step [100], Loss: 1.2472
Epoch [1/5], Step [200], Loss: 1.3565
Epoch [1/5], Step [300], Loss: 0.8937
Epoch [1/5], Step [400], Loss: 1.0873
Epoch [1/5], Step [500], Loss: 0.8764
Epoch [1/5], Step [600], Loss: 1.0200
Epoch [1/5], Step [700], Loss: 0.9107
Epoch [1/5], Step [800], Loss: 0.6019
Epoch [1/5], Step [900], Loss: 0.6863
Epoch [1/5], Step [1000], Loss: 0.7588
Epoch [1/5], Average Loss: 1.0518
Epoch [2/5], Step [100], Loss: 1.0521
Epoch [2/5], Step [200], Loss: 0.8057
Epoch [2/5], Step [300], Loss: 0.5568
Epoch [2/5], Step [400], Loss: 0.5241
Epoch [2/5], Step [500], Loss: 0.7578
Epoch [2/5], Step [600], Loss: 0.6290
Epoch [2/5], Step [700], Loss: 0.7354
Epoch [2/5], Step [800], Loss: 0.8706
Epoch [2/5], Step [900], Loss: 0.8033
Epoch [2/5], Step [1000], Loss: 0.4844
Epoch [2/5], Average Loss: 0.7823
Epoch [3/5], Step [100], Loss: 0.8579
Epoch [3/5], Step [200], Loss: 0.5118
Epoch [3/5], Step [300], Loss: 0.7818
Epoch [3/5], Step [400], Loss: 1.2262
Epoch [3/5], Step 

In [3]:
# Load the model later
model = ProtoEncoder().to(device)
model.load_state_dict(torch.load("proto_network.pth"))
model.eval()
print("Model loaded successfully.")


Model loaded successfully.


In [5]:
# Evaluation function for the Prototypical Network
def evaluate_prototypical_network(model, n_way=5, k_shot=5, q_query=15, device='cuda'):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    eval_loader = DataLoader(FewShotDataset(cifar10_test, n_way, k_shot, q_query), batch_size=1, shuffle=True)

    model.eval()  # Set model to evaluation mode

    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation for evaluation
        for support_images, query_images, labels in eval_loader:
            support_images, query_images, labels = (
                support_images.squeeze(0).to(device),
                query_images.squeeze(0).to(device),
                labels.to(device)
            )

            # Encode support and query images
            support_embeddings = model(support_images)  # (n_way * k_shot, embedding_dim)
            query_embeddings = model(query_images)      # (n_way * q_query, embedding_dim)

            # Reshape support embeddings and calculate prototypes (class means)
            support_embeddings = support_embeddings.view(n_way, k_shot, -1)
            prototypes = support_embeddings.mean(dim=1)  # (n_way, embedding_dim)

            # Compute distances between query embeddings and prototypes
            dists = torch.stack([euclidean_distance(query_embeddings, proto) for proto in prototypes], dim=1)

            # Predict class with the minimum distance (negative for similarity)
            preds = torch.argmin(dists, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    print(f"Evaluation Accuracy: {accuracy:.2f}%")

# Load the trained model for evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ProtoEncoder().to(device)

# Load model weights
model.load_state_dict(torch.load("proto_network.pth"))
print("Model loaded successfully.")

# Evaluate the model
evaluate_prototypical_network(model, device=device)


Model loaded successfully.
Evaluation Accuracy: 5762.20%
